# Garment Worker Tracking (YOLO11 Pose + ByteTrack)
Video upload korun -> per person **body tracking, hand (wrist) movement, working time** -> output **annotated video + JSON**

**Age:** Runtime -> Change runtime type -> **T4 GPU**

In [1]:
!pip install -q ultralytics

## 1) Video upload

In [2]:
from google.colab import files
up = files.upload()
VIDEO_PATH = list(up.keys())[0]
print("Video:", VIDEO_PATH)

Saving 0930.mp4 to 0930 (1).mp4
Video: 0930 (1).mp4


## 2) Tracking + analysis

In [3]:
import cv2, json, numpy as np
from collections import deque
from ultralytics import YOLO

# ================= SETTINGS (dorkar hole change korben) =================
MODEL_NAME   = "yolo11m-pose.pt"   # fast: yolo11n-pose.pt | accurate: yolo11l-pose.pt
CONF         = 0.35
IMGSZ        = 960
VID_STRIDE   = 1        # 2 dile ekta frame skip korbe (fast, kintu kom accurate)
MOVE_THRESH  = 0.006    # hand movement threshold (besi dile 'working' kom dekhabe)
WINDOW_SEC   = 1.0      # koto second-er average movement dekhe working/idle decide
MIN_SEEN_SEC = 2.0      # er cheye kom time dekha gele person ignore (false detection)
KP_CONF      = 0.5      # wrist keypoint confidence
OUT_JSON     = "output_report.json"
# =======================================================================

# tracker config (occlusion-e ID dhore rakhar jonno track_buffer boro)
open("custom_bytetrack.yaml", "w").write(
    "tracker_type: bytetrack\n"
    "track_high_thresh: 0.25\n"
    "track_low_thresh: 0.1\n"
    "new_track_thresh: 0.25\n"
    "track_buffer: 90\n"
    "match_thresh: 0.8\n"
    "fuse_score: True\n"
)

cap = cv2.VideoCapture(VIDEO_PATH)
fps = cap.get(cv2.CAP_PROP_FPS) or 25
W, H = int(cap.get(3)), int(cap.get(4))
total_frames = int(cap.get(7))
cap.release()

dt = VID_STRIDE / fps
out_fps = fps / VID_STRIDE
win = max(1, int(WINDOW_SEC * out_fps))
writer = cv2.VideoWriter("raw_out.mp4", cv2.VideoWriter_fourcc(*"mp4v"), out_fps, (W, H))

def hms(sec):
    sec = int(round(sec))
    return f"{sec//3600:02d}:{(sec%3600)//60:02d}:{sec%60:02d}"

def color_for(pid):
    rng = np.random.RandomState(pid * 7 + 3)
    return tuple(int(x) for x in rng.randint(60, 255, 3))

class PersonStat:
    def __init__(self, t):
        self.first_t = t
        self.last_t = t
        self.frames = 0
        self.work_time = 0.0
        self.prev = {"L": None, "R": None}
        self.dist_px = {"L": 0.0, "R": 0.0}
        self.dist_norm = {"L": 0.0, "R": 0.0}
        self.motion = deque(maxlen=win)
        self.trail = {"L": deque(maxlen=25), "R": deque(maxlen=25)}
        self.active = False
        self.timeline = []   # (time, active)

def work_segments(timeline, dt, merge_gap=1.0):
    segs, start, last = [], None, None
    for t, a in timeline:
        if a:
            if start is None:
                start = t
            elif t - last > merge_gap:
                segs.append([start, last + dt]); start = t
            last = t
    if start is not None:
        segs.append([start, last + dt])
    return [{"start_sec": round(a, 2), "end_sec": round(b, 2),
             "start": hms(a), "end": hms(b)} for a, b in segs]

model = YOLO(MODEL_NAME)
stats = {}

results = model.track(
    source=VIDEO_PATH, stream=True, persist=True,
    tracker="custom_bytetrack.yaml", conf=CONF, imgsz=IMGSZ,
    classes=[0], vid_stride=VID_STRIDE, verbose=False,
)

for i, r in enumerate(results):
    t = i * dt
    frame = r.plot(boxes=False, labels=False)      # body skeleton draw
    seen = set()

    if r.boxes is not None and r.boxes.id is not None and r.keypoints is not None:
        ids = r.boxes.id.int().cpu().tolist()
        boxes = r.boxes.xyxy.cpu().numpy()
        kxy = r.keypoints.xy.cpu().numpy()
        kcf = (r.keypoints.conf.cpu().numpy()
               if r.keypoints.conf is not None else np.ones(kxy.shape[:2]))

        for pid, box, kp, kc in zip(ids, boxes, kxy, kcf):
            seen.add(pid)
            x1, y1, x2, y2 = [int(v) for v in box]
            bh = max(y2 - y1, 1)
            s = stats.setdefault(pid, PersonStat(t))
            s.last_t = t + dt
            s.frames += 1

            frame_motion = 0.0
            for side, idx in (("L", 9), ("R", 10)):      # 9=left wrist, 10=right wrist
                if kc[idx] > KP_CONF:
                    p = kp[idx]
                    if s.prev[side] is not None:
                        d = float(np.linalg.norm(p - s.prev[side]))
                        s.dist_px[side] += d
                        s.dist_norm[side] += d / bh
                        frame_motion += d / bh
                    s.prev[side] = p
                    s.trail[side].append((int(p[0]), int(p[1])))
                else:
                    s.prev[side] = None

            s.motion.append(frame_motion)
            s.active = (sum(s.motion) / len(s.motion)) > MOVE_THRESH
            if s.active:
                s.work_time += dt
            s.timeline.append((round(t, 2), s.active))

            # ---- drawing ----
            col = color_for(pid)
            status_col = (0, 200, 0) if s.active else (0, 0, 220)
            cv2.rectangle(frame, (x1, y1), (x2, y2), col, 2)
            for side in ("L", "R"):
                pts = np.array(s.trail[side], dtype=np.int32)
                if len(pts) > 1:
                    cv2.polylines(frame, [pts], False, col, 2)
                if len(pts) > 0:
                    cv2.circle(frame, tuple(pts[-1]), 6, (0, 255, 255), -1)
            label = f"ID{pid} {'WORKING' if s.active else 'IDLE'} {hms(s.work_time)}"
            (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.6, 2)
            cv2.rectangle(frame, (x1, max(y1 - th - 10, 0)), (x1 + tw + 6, max(y1, th + 10)), status_col, -1)
            cv2.putText(frame, label, (x1 + 3, max(y1 - 5, th + 4)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)

    # jara ei frame-e dekha jayni, tader wrist state reset (jump bondho korte)
    for pid, s in stats.items():
        if pid not in seen:
            s.prev = {"L": None, "R": None}
            s.motion.clear()
            s.active = False

    cv2.putText(frame, f"Time {hms(t)}  People: {len(seen)}", (15, 35),
                cv2.FONT_HERSHEY_SIMPLEX, 1.0, (255, 255, 255), 3)
    writer.write(frame)

    if i % 100 == 0:
        print(f"frame {i * VID_STRIDE}/{total_frames}", end="\r")

writer.release()

# ================= JSON report =================
people = []
for pid, s in sorted(stats.items()):
    present = s.frames * dt
    if present < MIN_SEEN_SEC:
        continue
    work = s.work_time
    people.append({
        "person_id": pid,
        "first_seen_sec": round(s.first_t, 2),
        "last_seen_sec": round(s.last_t, 2),
        "present_time_sec": round(present, 2),
        "present_time": hms(present),
        "working_time_sec": round(work, 2),
        "working_time": hms(work),
        "idle_time_sec": round(present - work, 2),
        "idle_time": hms(present - work),
        "working_percent": round(100 * work / present, 1),
        "hand_movement": {
            "left_total_px": round(s.dist_px["L"], 1),
            "right_total_px": round(s.dist_px["R"], 1),
            "left_total_normalized": round(s.dist_norm["L"], 3),
            "right_total_normalized": round(s.dist_norm["R"], 3),
            "avg_normalized_per_sec": round((s.dist_norm["L"] + s.dist_norm["R"]) / present, 4),
        },
        "working_segments": work_segments(s.timeline, dt),
    })

report = {
    "video": {"file": VIDEO_PATH, "fps": round(fps, 2), "width": W, "height": H,
              "duration_sec": round(total_frames / fps, 2)},
    "settings": {"model": MODEL_NAME, "move_thresh": MOVE_THRESH,
                 "window_sec": WINDOW_SEC, "vid_stride": VID_STRIDE},
    "total_people": len(people),
    "people": people,
}
with open(OUT_JSON, "w") as f:
    json.dump(report, f, indent=2, ensure_ascii=False)

print("\nDone! Total people:", len(people))
for p in people:
    print(f"ID {p['person_id']}: present {p['present_time']} | working {p['working_time']} ({p['working_percent']}%)")


Done! Total people: 18
ID 1: present 00:00:03 | working 00:00:03 (92.9%)
ID 2: present 00:00:05 | working 00:00:04 (83.7%)
ID 3: present 00:00:06 | working 00:00:04 (62.9%)
ID 19: present 00:00:05 | working 00:00:04 (93.0%)
ID 21: present 00:00:03 | working 00:00:01 (45.7%)
ID 32: present 00:00:05 | working 00:00:04 (94.9%)
ID 40: present 00:00:02 | working 00:00:02 (92.9%)
ID 41: present 00:00:05 | working 00:00:05 (98.1%)
ID 42: present 00:00:02 | working 00:00:02 (78.6%)
ID 43: present 00:00:02 | working 00:00:01 (47.1%)
ID 44: present 00:00:02 | working 00:00:02 (98.6%)
ID 46: present 00:00:02 | working 00:00:02 (91.4%)
ID 48: present 00:00:02 | working 00:00:02 (68.6%)
ID 53: present 00:00:03 | working 00:00:03 (88.8%)
ID 59: present 00:00:03 | working 00:00:01 (40.9%)
ID 60: present 00:00:03 | working 00:00:03 (98.8%)
ID 61: present 00:00:03 | working 00:00:03 (98.9%)
ID 66: present 00:00:02 | working 00:00:02 (95.5%)


## 3) Video convert + Download

In [4]:
!ffmpeg -y -loglevel error -i raw_out.mp4 -vcodec libx264 -pix_fmt yuv420p output_tracked.mp4

from google.colab import files
files.download("output_tracked.mp4")
files.download("output_report.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>